# Literature Search Agent

修改下方的配置变量，然后依次运行每个单元格即可。

In [ ]:
# 首次使用：安装依赖（之后无需重复）
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "nest_asyncio", "-q"], check=True)
print("安装完成")

In [ ]:
# ── 搜索配置（修改这里）────────────────────────────────────────────

QUERY = "social capital in education"  # 关键词 或 研究问题

MODE = "balanced"        # "balanced" | "classic"（高被引）| "frontier"（2020+）

PERSPECTIVE = None       # None 或以下之一：
                         # "profession" "organization" "symbolic"
                         # "stratification" "network" "culture"

MAX_PAPERS = 50          # 最多返回篇数
INCLUDE_ABSTRACT = False # True = BibTeX 中包含摘要
SAVE_NOTION = False      # True = 同步写入 Notion（需配置 .env）
CLAUDE_ONLY = False      # True = 跳过学术 API，由 Claude 直接生成文献

OUTPUT_FILE = "results.bib"  # 输出文件名

print(f"查询: {QUERY}")
print(f"模式: {MODE}  |  视角: {PERSPECTIVE}  |  最多: {MAX_PAPERS} 篇")
print(f"离线模式: {CLAUDE_ONLY}  |  Notion: {SAVE_NOTION}")

In [ ]:
import asyncio
import nest_asyncio
nest_asyncio.apply()  # 让 asyncio.run() 可以在 Jupyter 内核中运行

from lit_search import agent

asyncio.run(
    agent.run(
        query=QUERY,
        output_path=OUTPUT_FILE,
        max_papers=MAX_PAPERS,
        include_abstract=INCLUDE_ABSTRACT,
        mode=MODE,
        perspective=PERSPECTIVE,
        save_notion=SAVE_NOTION,
        claude_only=CLAUDE_ONLY,
    )
)

In [ ]:
# 解析结果，显示文献列表表格
import re
from pathlib import Path

bib_text = Path(OUTPUT_FILE).read_text(encoding="utf-8")

entries = []
for block in re.split(r"\n(?=@)", bib_text):
    if not block.strip().startswith("@"):
        continue
    def _get(field):
        m = re.search(rf"{field}\s*=\s*\{{([^}}]{{0,300}})", block, re.IGNORECASE)
        return m.group(1).strip() if m else ""
    entries.append({
        "Title": _get("title")[:80],
        "Author": _get("author").split(" and ")[0][:30],
        "Year": _get("year"),
        "Journal": (_get("journal") or _get("booktitle"))[:40],
        "DOI": _get("doi")[:40],
    })

print(f"找到 {len(entries)} 篇文献\n")

try:
    import pandas as pd
    df = pd.DataFrame(entries)
    display(df)
except ImportError:
    # pandas 未安装时的纯文本输出
    header = f"{'#':<4} {'Year':<6} {'Author':<32} {'Title'}"
    print(header)
    print("-" * 100)
    for i, e in enumerate(entries, 1):
        print(f"{i:<4} {e['Year']:<6} {e['Author']:<32} {e['Title']}")

In [ ]:
# 显示 Claude 生成的文献综述摘要（如果有）
lines = bib_text.splitlines()
in_summary = False
summary_lines = []
for line in lines:
    if "LITERATURE LANDSCAPE" in line:
        in_summary = True
        continue
    if in_summary:
        summary_lines.append(line.lstrip("% "))

if summary_lines:
    from IPython.display import Markdown, display
    display(Markdown("### 文献综述\n\n" + "\n".join(summary_lines)))
else:
    print("无摘要（需要设置 ANTHROPIC_API_KEY）")

In [ ]:
# 预览 BibTeX 原文（前 60 行）
preview_lines = bib_text.splitlines()[:60]
print("\n".join(preview_lines))
print(f"\n... 完整内容见 {OUTPUT_FILE}")